# Clase 4 — Programación lineal: problema de producción

Notebook de referencia (demo en vivo) para acompañar `slides/clase4.md`.

Formulamos y resolvemos el problema de producción de sillas y mesas
(ver slides) con `scipy.optimize.linprog`, y visualizamos la región
factible con el óptimo marcado.

**Problema:**
- Maximizar 20·x1 + 30·x2 (ganancia: sillas, mesas)
- sujeto a: 2·x1 + 3·x2 ≤ 120 (madera)
-           x1 + x2 ≤ 50   (horas de mano de obra)
-           x1, x2 ≥ 0


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog


## Formulación (linprog minimiza, así que negamos la función objetivo)

In [ ]:
# linprog resuelve problemas de MINIMIZACION, así que para maximizar
# 20 x1 + 30 x2 minimizamos -(20 x1 + 30 x2)
c = [-20, -30]

# Restricciones de la forma A_ub @ x <= b_ub
A_ub = [
    [2, 3],   # madera
    [1, 1],   # mano de obra
]
b_ub = [120, 50]

bounds = [(0, None), (0, None)]  # x1, x2 >= 0

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
print(res)
print(f"\nÓptimo: x1={res.x[0]:.2f} sillas, x2={res.x[1]:.2f} mesas")
print(f"Ganancia máxima: ${-res.fun:.2f}")


## Visualización de la región factible

In [ ]:
x1 = np.linspace(0, 60, 400)

# despejamos x2 de cada restricción
x2_madera = (120 - 2 * x1) / 3
x2_labor = 50 - x1

x2_upper = np.minimum(x2_madera, x2_labor)
x2_upper = np.clip(x2_upper, 0, None)

plt.figure(figsize=(6, 6))
plt.plot(x1, x2_madera, label="2x1 + 3x2 = 120 (madera)")
plt.plot(x1, x2_labor, label="x1 + x2 = 50 (mano de obra)")
plt.fill_between(x1, 0, x2_upper, where=(x2_upper >= 0), alpha=0.3, label="Región factible")

plt.scatter([res.x[0]], [res.x[1]], color="red", zorder=5, label="Óptimo")

plt.xlim(0, 60)
plt.ylim(0, 50)
plt.xlabel("x1 (sillas)")
plt.ylabel("x2 (mesas)")
plt.title("Región factible y óptimo del problema de producción")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Para la actividad de los estudiantes

- Resuelvan el mismo problema **a mano**, evaluando la función
  objetivo en cada vértice de la región factible, y comprueben que
  coincide con lo que devuelve `linprog`.
- Cambien los coeficientes (ganancia por silla/mesa, disponibilidad
  de madera y horas) y observen cómo se mueve el vértice óptimo.
- (Bono TSP) intenten formular con `scipy.optimize.linprog` un
  problema de mezcla/dieta con 3+ variables y 3+ restricciones.
